# Baseline insolvency model

Label = `CompanyStatus` (Sam's `classify_status`).
Features = `shap_feature_matrix_with_insolvencies.parquet` (the matrix rebuilt to keep insolvent firms).
Models = Logistic Regression + Gradient Boosting, explained with SHAP.


In [ ]:
# Install deps, then RESTART the kernel once, then Run All.
# numpy pinned <2 + pyarrow <16 to match this conda env's compiled packages (fixes _ARRAY_API warnings).
%pip install "numpy<2" "pyarrow<16" shap scikit-learn matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO   = r"C:/MSC/Project/lloyds-commercial-banking-intelligence-2026"
# Single source: the matrix that KEEPS insolvent companies (has CompanyStatus + all features).
MATRIX = Path(REPO + "/data/processed/shap_feature_matrix_with_insolvencies.parquet")

In [ ]:
# Sam's status -> insolvency label (adapted from samuel/attrition).
_INSOLVENCY = {"liquidation", "in administration", "in administration/administrative receiver",
               "administration order", "voluntary arrangement",
               "live but receiver manager on at least one charge", "receivership"}

def is_insolvent(status):
    s = str(status).strip().lower()
    return int(s in _INSOLVENCY or "administration" in s or "liquidation" in s)

## Load the matrix and label (single file - no merge)

In [ ]:
df = pd.read_parquet(MATRIX)
df.columns = df.columns.str.strip()

df["y"] = df["CompanyStatus"].map(is_insolvent)
# keep insolvent vs active only
df = df[df["CompanyStatus"].str.strip().str.lower().eq("active") | (df["y"] == 1)].copy()

print("rows:", len(df), "| insolvent:", int(df["y"].sum()), f"({df['y'].mean():.2%})")

## Build the feature matrix

Static features are already in the matrix, computed for **all** companies (incl. insolvent). We keep the
honest ones and deliberately drop the leaky ones:
- `accounts_overdue`, `accounts_stale`, `financial_health_concern` -> insolvency *symptoms* (firms stop filing).
- `api_enriched` + the API/contract columns -> real only for the ~1000 active-prioritised sample, so they
  carry ~no signal for insolvency and would re-introduce an "is-active" leak.


In [ ]:
FEATURES = [
    "company_age_years", "debt_ratio",
    "has_any_charge", "has_outstanding_charges", "has_name_change", "num_previous_names",
    "Mortgages.NumMortCharges", "Mortgages.NumMortOutstanding",
    "Mortgages.NumMortSatisfied", "Mortgages.NumMortPartSatisfied",
]

X = df[FEATURES].apply(pd.to_numeric, errors="coerce")
X = pd.concat([
    X,
    pd.get_dummies(df["sector"],    prefix="sector", dummy_na=True).astype(float),
    pd.get_dummies(df["size_tier"], prefix="size",   dummy_na=True).astype(float),
], axis=1).astype("float64")
y = df["y"].astype(int)
print("feature matrix:", X.shape)

## Split + Logistic Regression (interpretable)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

logit = Pipeline([
    ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
logit.fit(X_tr, y_tr)
pd.Series(logit.named_steps["clf"].coef_[0], index=X.columns).sort_values().round(3)

## Gradient Boosting (complex)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

w = np.where(y_tr == 1, (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1), 1.0)
gb = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05, random_state=42)
gb.fit(X_tr, y_tr, sample_weight=w)
print("trained")

## SHAP explanations (falls back to coef / permutation importance if shap won't import)

In [ ]:
try:
    import shap
    sc = logit.named_steps["scale"]
    X_te_imp = pd.DataFrame(logit.named_steps["impute"].transform(X_te), columns=X.columns, index=X_te.index).values
    sv = shap.LinearExplainer(logit.named_steps["clf"], sc.transform(X_te_imp)).shap_values(sc.transform(X_te_imp))
    imp_logit = pd.Series(np.abs(sv).mean(0), index=X.columns)
    svg = shap.TreeExplainer(gb).shap_values(X_te)
    svg = svg[1] if isinstance(svg, list) else svg
    imp_gb = pd.Series(np.abs(svg).mean(0), index=X.columns)
    print("SHAP importances (mean |SHAP value|):")
except Exception as e:
    from sklearn.inspection import permutation_importance
    print("shap unavailable ->", type(e).__name__, "- using coefficient / permutation importance instead.")
    imp_logit = pd.Series(np.abs(logit.named_steps["clf"].coef_[0]), index=X.columns)
    pi = permutation_importance(gb, X_te.fillna(X_te.median()), y_te, n_repeats=5, random_state=42)
    imp_gb = pd.Series(pi.importances_mean, index=X.columns)

print("\nLogistic - top features:")
print(imp_logit.sort_values(ascending=False).head(10).round(4))
print("\nGradient Boosting - top features:")
print(imp_gb.sort_values(ascending=False).head(10).round(4))

## Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def prec_at_k(y_true, score, frac=0.10):
    n = max(1, int(len(score) * frac))
    top = np.argsort(score)[::-1][:n]
    return np.asarray(y_true)[top].mean()

rows = []
for name, m in [("LogisticRegression", logit), ("GradientBoosting", gb)]:
    p = m.predict_proba(X_te)[:, 1]
    rows.append({"model": name, "roc_auc": roc_auc_score(y_te, p),
                 "pr_auc": average_precision_score(y_te, p),
                 "precision@10%": prec_at_k(y_te, p)})
res = pd.DataFrame(rows).set_index("model")
res["base_rate"] = y.mean()
res.round(3)

> The old missingness leak is fixed: insolvent firms are now rows in the matrix with real static
> features, so the models rely on genuine signal (company age, charges, debt ratio, sector) rather than a
> NaN pattern. Expect realistic ROC-AUC/PR-AUC now, not ~0.99.